In [1]:
from jupyter_dash import JupyterDash
import dash_leaflet as dl
from dash import dcc, html, dash_table
from dash.dependencies import Input, Output
import plotly.express as px
import pandas as pd
import base64
from CRUD_Python_Module import AnimalShelter

############## Data Manipulation / Model

username = "aacuser"
password = "aacuser123"
shelter = AnimalShelter(username, password)

# Load data and handle empty results
data = shelter.read({})
if not data:
    print("⚠ WARNING: No data in database!")
    df = pd.DataFrame()
else:
    df = pd.DataFrame(data)
    if '_id' in df.columns:
        df.drop(columns=['_id'], inplace=True)
    print(f"✓ Loaded {len(df)} animals")

# Load logo
logo_path = 'Grazioso Salvare Logo.png'
try:
    with open(logo_path, 'rb') as f:
        logo_base64 = base64.b64encode(f.read()).decode('ascii')
        logo_src = f'data:image/png;base64,{logo_base64}'
    print("✓ Logo loaded")
except FileNotFoundError:
    logo_src = ''
    print("⚠ Logo not found")

################ Dashboard Layout / View

app = JupyterDash('SimpleExample')

app.layout = html.Div([
    html.Div(id='hidden-div', style={'display': 'none'}),
    
    # Logo (links to SNHU)
    html.A(
        html.Img(src=logo_src, style={'height': '80px'}),
        href='https://www.snhu.edu',
        target='_blank')
    if logo_src else None,
    
    html.H1("🐾 Elle Ward — Grazioso Salvare Animal Rescue Dashboard"),
    html.Hr(),
    
    # Radio buttons for filtering
    html.H3("Select Rescue Type:"),
    dcc.RadioItems(
        id='filter-type',
        options=[
            {'label': ' Reset', 'value': 'reset'},
            {'label': ' Water Rescue', 'value': 'water'},
            {'label': ' Mountain/Wilderness', 'value': 'mountain'},
            {'label': ' Disaster Tracking', 'value': 'disaster'}],
        value='reset',
        inline=True,
        labelStyle={'marginRight': '20px'}),
    html.Hr(),
    
    # Data table
    dash_table.DataTable(
        id='datatable-id',
        columns=[
            {"name": i, "id": i, "deletable": False, "selectable": True}
            for i in df.columns]
        if len(df) > 0 else [],
        data=df.to_dict('records') if len(df) > 0 else [],
        row_selectable="single",
        selected_rows=[0],
        page_action="native",
        page_size=10,
        filter_action="native",
        sort_action="native",
        style_table={'overflowX': 'auto'},
        style_cell={'textAlign': 'left'},
    ),
    html.Br(),
    html.Hr(),
    
    # Map and chart side by side
    html.Div([
        html.Div(id='map-id', className='col s12 m6', 
                 style={'width': '48%', 'display': 'inline-block'}),
        html.Div([
            html.H3("📊 Breed Distribution", style={'textAlign': 'center'}),
            dcc.Graph(id='breed-chart')
        ], style={'width': '48%', 'display': 'inline-block', 'marginLeft': '4%', 'textAlign': 'center'})
    ])
])

######################## Interaction Between Components/Controller

# Filter callback to update table when radio button changes
@app.callback(
    Output('datatable-id', 'data'),
    [Input('filter-type', 'value')])

def update_dashboard(filter_type):
    if filter_type == 'water':
        query = shelter.filter_water_rescue()
    elif filter_type == 'mountain':
        query = shelter.filter_mountain_wilderness_rescue()
    elif filter_type == 'disaster':
        query = shelter.filter_disaster_tracking()
    else:
        query = shelter.filter_reset()
    
    data = shelter.read(query)
    df_filtered = pd.DataFrame(data)
    if '_id' in df_filtered.columns:
        df_filtered.drop(columns=['_id'], inplace=True)
    return df_filtered.to_dict('records')


# Column highlighting
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')])

def update_styles(selected_columns):
    if not selected_columns:
        return []
    return [{
        'if': {'column_id': i},
        'background_color': '#D2F3FF'}
        for i in selected_columns]


# Map 
@app.callback(
    Output('map-id', 'children'),
    [Input('datatable-id', 'derived_virtual_data'),
     Input('datatable-id', 'derived_virtual_selected_rows')])

def update_map(viewData, index):
    if viewData is None or len(viewData) == 0:
        return []
    dff = pd.DataFrame.from_dict(viewData)
    row = 0 if index is None or len(index) == 0 else index[0]
    return [
        dl.Map(
            style={'width': '1000px', 'height': '500px'},
            center=[30.75, -97.48],
            zoom=10,
            children=[
                dl.TileLayer(id="base-layer-id"),
                dl.Marker(
                    position=[dff.iloc[row, 13], dff.iloc[row, 14]],
                    children=[
                        dl.Tooltip(dff.iloc[row, 4]),
                        dl.Popup([
                            html.H1("Animal Name"),
                            html.P(dff.iloc[row, 9])
                        ])
                    ]
                )
            ]
        )
    ]


# Breed chart 
@app.callback(
    Output('breed-chart', 'figure'),
    [Input('datatable-id', 'derived_virtual_data')])

def update_breed_chart(viewData):
    if viewData is None or len(viewData) == 0:
        return {'data': [], 'layout': {'title': 'No data'}}
    
    dff = pd.DataFrame.from_dict(viewData)
    breed_counts = dff['breed'].value_counts().head(10)
    
    fig = px.pie(
        values=breed_counts.values,
        names=breed_counts.index,
        title=f'Top 10 Breeds ({len(dff)} dogs)',
        hole=0.3)
    return fig


app.run_server(debug=True, host='0.0.0.0', port=8051)

✓ Loaded 10000 animals
✓ Logo loaded
Dash app running on http://0.0.0.0:8051/
